# AlexNet on ImageNet

Load a pretrained AlexNet, run it on a handful of images pulled from the internet (each with a known ground-truth ImageNet class), and visualize predictions — including the misclassified ones.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import requests
import torch
from PIL import Image
from torchvision.models import AlexNet_Weights, alexnet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
weights = AlexNet_Weights.IMAGENET1K_V1
model = alexnet(weights=weights).to(device).eval()
preprocess = weights.transforms()
categories = weights.meta["categories"]  # 1000 human-readable ImageNet class names
len(categories), categories[:5]

## Test images

A small curated set of Wikimedia Commons images, each paired with its true ImageNet class. Mix of easy, canonical shots and a few harder / atypical ones to coax out some misclassifications.

In [ ]:
# (image URL, ground-truth human-readable label)
TEST_IMAGES = [
    (
        "https://commons.wikimedia.org/wiki/Special:FilePath/Golden_Retriever_medium-to-light-coat.jpg",
        "golden retriever",
    ),
    ("https://commons.wikimedia.org/wiki/Special:FilePath/Cat03.jpg", "tabby cat"),
    ("https://commons.wikimedia.org/wiki/Special:FilePath/African_Bush_Elephant.jpg", "African elephant"),
    ("https://commons.wikimedia.org/wiki/Special:FilePath/Plains_Zebra_Equus_quagga.jpg", "zebra"),
    ("https://commons.wikimedia.org/wiki/Special:FilePath/Koala_climbing_tree.jpg", "koala"),
    ("https://commons.wikimedia.org/wiki/Special:FilePath/Red_Apple.jpg", "Granny Smith"),
    ("https://commons.wikimedia.org/wiki/Special:FilePath/Phare_de_Cordouan.jpg", "beacon"),
    (
        "https://commons.wikimedia.org/wiki/Special:FilePath/00_Walden_Acoustic_Guitar_D310e_body.jpg",
        "acoustic guitar",
    ),
    ("https://commons.wikimedia.org/wiki/Special:FilePath/Steam_locomotive.jpg", "steam locomotive"),
    ("https://commons.wikimedia.org/wiki/Special:FilePath/Hot_air_balloon_and_moon.jpg", "balloon"),
    ("https://commons.wikimedia.org/wiki/Special:FilePath/Siberian_Husky.jpg", "Siberian husky"),
    ("https://commons.wikimedia.org/wiki/Special:FilePath/Mandarin_Duck.jpg", "drake"),
]

IMG_DIR = Path("images")
IMG_DIR.mkdir(exist_ok=True)

HEADERS = {"User-Agent": "AlexNetDemoNotebook/1.0 (educational use; contact neworldemancer@gmail.com)"}


def download(url: str, dest: Path) -> Path:
    if not dest.exists():
        resp = requests.get(url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        dest.write_bytes(resp.content)
    return dest


samples = []
for url, true_label in TEST_IMAGES:
    fname = IMG_DIR / url.rsplit("/", 1)[-1]
    try:
        download(url, fname)
        img = Image.open(fname).convert("RGB")
        samples.append((img, true_label, fname.name))
    except Exception as e:
        print(f"skip {fname.name}: {e}")

len(samples)

## Inference

In [ ]:
results = []
with torch.no_grad():
    for img, true_label, fname in samples:
        batch = preprocess(img).unsqueeze(0).to(device)
        logits = model(batch)
        probs = logits.softmax(dim=1)[0]
        top_idx = int(probs.argmax())
        pred_label = categories[top_idx]
        confidence = float(probs[top_idx])
        correct = true_label.lower() in pred_label.lower() or pred_label.lower() in true_label.lower()
        results.append(
            {
                "img": img,
                "fname": fname,
                "true_label": true_label,
                "pred_label": pred_label,
                "confidence": confidence,
                "correct": correct,
            }
        )

for r in results:
    status = "OK" if r["correct"] else "MISS"
    print(
        f"[{status}] {r['fname']}: true={r['true_label']!r} pred={r['pred_label']!r} ({r['confidence']:.1%})"
    )

## All predictions, overlaid on the images

In [ ]:
def plot_grid(items, ncols=4, title=None):
    n = len(items)
    if n == 0:
        print("(none)")
        return
    ncols = min(ncols, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4.5 * nrows))
    axes = axes.flatten() if n > 1 else [axes]
    for ax, r in zip(axes, items):
        ax.imshow(r["img"])
        ax.axis("off")
        color = "limegreen" if r["correct"] else "red"
        caption = f"pred: {r['pred_label']} ({r['confidence']:.0%})\ntrue: {r['true_label']}"
        ax.set_title(caption, color=color, fontsize=10)
    for ax in axes[n:]:
        ax.axis("off")
    if title:
        fig.suptitle(title, fontsize=14)
    fig.tight_layout()
    plt.show()


plot_grid(results, title="AlexNet predictions (green = correct, red = miss)")

## Misclassified examples

In [ ]:
misses = [r for r in results if not r["correct"]]
print(f"{len(misses)}/{len(results)} misclassified")
plot_grid(misses, title="Misclassified")

## Robustness to input perturbations

For every test image, apply three kinds of perturbation — **scale** (digital zoom in/out), **brightness**, and **contrast** — each swept on a log scale around the identity (factor = 100%), and track how the predicted probability of the *ground-truth* class changes.

In [ ]:
import numpy as np
from PIL import ImageEnhance


def zoom(img: Image.Image, factor: float) -> Image.Image:
    """Digitally zoom in (factor>1) or out (factor<1) while keeping canvas size fixed.

    Crops first and resizes after (rather than resizing the whole image up before
    cropping) so a factor like 3.0 on a large photo never materializes a
    multi-hundred-megapixel intermediate.
    """
    W, H = img.size
    if factor >= 1:
        crop_w, crop_h = max(1, round(W / factor)), max(1, round(H / factor))
        left, top = (W - crop_w) // 2, (H - crop_h) // 2
        cropped = img.crop((left, top, left + crop_w, top + crop_h))
        return cropped.resize((W, H), Image.BILINEAR)
    else:
        new_w, new_h = max(1, round(W * factor)), max(1, round(H * factor))
        scaled = img.resize((new_w, new_h), Image.BILINEAR)
        canvas = Image.new("RGB", (W, H), (0, 0, 0))
        canvas.paste(scaled, ((W - new_w) // 2, (H - new_h) // 2))
        return canvas


# (display name, transform fn, factors swept log-uniformly around identity=1.0)
TRANSFORMS = [
    ("scale", zoom, np.geomspace(0.30, 3.00, 13)),
    ("brightness", lambda img, f: ImageEnhance.Brightness(img).enhance(f), np.geomspace(0.10, 3.00, 13)),
    ("contrast", lambda img, f: ImageEnhance.Contrast(img).enhance(f), np.geomspace(0.10, 3.00, 13)),
]


def gt_class_index(true_label: str) -> int:
    for i, c in enumerate(categories):
        if true_label.lower() in c.lower() or c.lower() in true_label.lower():
            return i
    raise ValueError(f"no ImageNet category matches {true_label!r}")


sweep = {name: {} for name, _, _ in TRANSFORMS}
with torch.no_grad():
    for name, fn, factors in TRANSFORMS:
        for img, true_label, fname in samples:
            gt_idx = gt_class_index(true_label)
            probs = []
            for f in factors:
                timg = fn(img, float(f))
                batch = preprocess(timg).unsqueeze(0).to(device)
                p = model(batch).softmax(dim=1)[0]
                probs.append(float(p[gt_idx]))
            sweep[name][fname] = probs

print("done:", {name: len(d) for name, d in sweep.items()})

In [ ]:
fig, axes = plt.subplots(len(TRANSFORMS), 1, figsize=(9, 4 * len(TRANSFORMS)))
colors = plt.cm.tab20(np.linspace(0, 1, len(samples)))

for ax, (name, _, factors) in zip(axes, TRANSFORMS):
    for (img, true_label, fname), color in zip(samples, colors):
        ax.plot(
            factors * 100, sweep[name][fname], marker="o", ms=3, color=color, label=f"{true_label} ({fname})"
        )
    ax.set_xscale("log")
    ax.axvline(100, color="black", ls="--", lw=1, alpha=0.6, label="identity")
    ax.set_xlabel(f"{name} (%)")
    ax.set_ylabel("P(true class)")
    ax.set_ylim(0, 1.02)
    ax.set_title(f"Ground-truth class probability vs. {name}")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, fontsize=7, loc="center left", bbox_to_anchor=(1.0, 0.5))
fig.tight_layout()
plt.show()

### The actual transformed images

For each test image: one figure, 3 rows (scale / brightness / contrast), each column a step along that transform's sweep — subplot title shows the parameter value and the resulting P(true class).

In [ ]:
DISPLAY_SIZE = 300  # downsize before transforming for display only - keeps plotting fast

for img, true_label, fname in samples:
    thumb = img.copy()
    thumb.thumbnail((DISPLAY_SIZE, DISPLAY_SIZE), Image.BILINEAR)

    fig, axes = plt.subplots(
        len(TRANSFORMS), len(TRANSFORMS[0][2]), figsize=(1.6 * len(TRANSFORMS[0][2]), 1.8 * len(TRANSFORMS))
    )
    for row, (name, fn, factors) in zip(axes, TRANSFORMS):
        probs = sweep[name][fname]
        for ax, factor, prob in zip(row, factors, probs):
            ax.imshow(fn(thumb, float(factor)))
            ax.axis("off")
            ax.set_title(f"{name} {factor * 100:.0f}%\nP={prob:.0%}", fontsize=7)
    fig.suptitle(f"{fname} (true: {true_label})", fontsize=11)
    fig.tight_layout()
    plt.show()
    plt.close(fig)

## Pushing misclassified inputs toward the correct class

For every image AlexNet got wrong, run many steps of gradient ascent directly on the input pixels (image fixed, weights fixed) to maximize the logit of the *true* class, and show the resulting image after each update.

In [ ]:
N_STEPS = 1000
SNAPSHOT_EVERY = 200
STEP_SIZE = 0.05

mean = torch.tensor(preprocess.mean).view(3, 1, 1).to(device)
std = torch.tensor(preprocess.std).view(3, 1, 1).to(device)

FRAMES_DIR = IMG_DIR / "opt_study" / "frames"
FRAMES_DIR.mkdir(parents=True, exist_ok=True)


def tensor_to_pil(x: torch.Tensor) -> Image.Image:
    img = (x.detach().cpu() * std.cpu() + mean.cpu()).clamp(0, 1)
    return Image.fromarray((img.permute(1, 2, 0).numpy() * 255).astype("uint8"))


def save_frame(frame: Image.Image, stem: str, mode: str, step: int) -> None:
    """Persist each snapshot to disk so later replots (e.g. a different SNAPSHOT_EVERY
    or layout) don't require rerunning the optimization."""
    frame.save(FRAMES_DIR / f"{stem}__{mode}__step{step:04d}.png")


opt_results = []  # one entry per misclassified image: dict(fname, true_label, frames, probs, steps)
for r in misses:
    gt_idx = gt_class_index(r["true_label"])
    stem = Path(r["fname"]).stem
    x = preprocess(r["img"]).clone().to(device).requires_grad_(True)

    steps, frames, probs = [], [], []
    for step in range(N_STEPS + 1):
        logits = model(x.unsqueeze(0))
        prob_gt = logits.softmax(dim=1)[0, gt_idx]
        if step % SNAPSHOT_EVERY == 0 or step == N_STEPS:
            frame = tensor_to_pil(x)
            steps.append(step)
            frames.append(frame)
            probs.append(float(prob_gt.detach()))
            save_frame(frame, stem, "plain", step)
        if step == N_STEPS:
            break
        grad = torch.autograd.grad(logits[0, gt_idx], x)[0]
        with torch.no_grad():
            x += STEP_SIZE * grad / grad.norm()

    opt_results.append(
        {"fname": r["fname"], "true_label": r["true_label"], "steps": steps, "frames": frames, "probs": probs}
    )
    print(
        f"{r['fname']}: true={r['true_label']!r}  P(true) {probs[0]:.1%} -> {probs[-1]:.1%}  "
        f"over {N_STEPS} steps (shown every {SNAPSHOT_EVERY})"
    )

In [ ]:
n_rows = len(opt_results)
n_steps_cols = len(opt_results[0]["steps"])
n_cols = n_steps_cols + 3  # + initial repeated (start-vs-end), + raw diff, + amplified diff
fig, axes = plt.subplots(n_rows, n_cols, figsize=(1.6 * n_cols, 1.8 * n_rows))
if n_rows == 1:
    axes = axes[None, :]

for row, r in zip(axes, opt_results):
    step_axes = row[:n_steps_cols]
    again_ax, diff_ax, amp_ax = row[n_steps_cols], row[n_steps_cols + 1], row[n_steps_cols + 2]

    for ax, step, frame, prob in zip(step_axes, r["steps"], r["frames"], r["probs"]):
        ax.imshow(frame)
        ax.axis("off")
        ax.set_title(f"step {step}\nP={prob:.0%}", fontsize=8)

    # initial frame shown again right after the final step, so start and end sit
    # side by side instead of at opposite ends of the row
    again_ax.imshow(r["frames"][0])
    again_ax.axis("off")
    again_ax.set_title(f"step {r['steps'][0]} (again)\nP={r['probs'][0]:.0%}", fontsize=8)

    initial = np.asarray(r["frames"][0], dtype=np.float32) / 255.0
    final = np.asarray(r["frames"][-1], dtype=np.float32) / 255.0
    diff = np.abs(final - initial)

    diff_ax.imshow(diff, vmin=0, vmax=1)
    diff_ax.axis("off")
    diff_ax.set_title(f"|Δ| (0-1 scale)\nmax={diff.max():.2f}", fontsize=8)

    d_min, d_max = diff.min(), diff.max()
    amplified = (diff - d_min) / (d_max - d_min + 1e-12)
    amp_ax.imshow(amplified, vmin=0, vmax=1)
    amp_ax.axis("off")
    amp_ax.set_title(f"|Δ| amplified\n[{d_min:.2f}, {d_max:.2f}]→[0,1]", fontsize=8)

    row[0].text(
        -0.15,
        0.5,
        f"{r['true_label']}",
        transform=row[0].transAxes,
        ha="right",
        va="center",
        fontsize=9,
        rotation=0,
    )

fig.suptitle(
    f"Gradient ascent on input pixels toward the true class ({N_STEPS} steps, shown every {SNAPSHOT_EVERY})",
    fontsize=11,
)
fig.tight_layout()
plt.show()
plt.close(fig)

## Same optimization, but with random augmentation each step (EOT-style)

Identical setup to the previous section, except the gradient driving each update is computed on a *randomly augmented* view of the current image rather than the image itself: scale ±10% (+ implicit crop/pad back to canvas size), rotation ±10°, brightness/contrast/saturation ±10%, hue ±10%. The tracked probability is still that of the actual (unaugmented) optimized image, so it's directly comparable to the previous section.

In [ ]:
import random

import torchvision.transforms as T
import torchvision.transforms.functional as TF

N_AUG_SAMPLES = 32  # gradient averaged over this many random augmentations per step (EOT)


def random_augment(x: torch.Tensor) -> torch.Tensor:
    """Random scale+rotation+color jitter, applied in [0,1] pixel space, returned normalized."""
    img01 = (x * std + mean).clamp(0, 1)
    img01 = TF.affine(
        img01,
        angle=random.uniform(-10, 10),
        translate=[0, 0],
        scale=random.uniform(0.9, 1.1),
        shear=[0.0, 0.0],
        interpolation=T.InterpolationMode.BILINEAR,
    )
    img01 = TF.adjust_brightness(img01, random.uniform(0.9, 1.1))
    img01 = TF.adjust_contrast(img01, random.uniform(0.9, 1.1))
    img01 = TF.adjust_saturation(img01, random.uniform(0.9, 1.1))
    img01 = TF.adjust_hue(img01, random.uniform(-0.1, 0.1))
    return (img01 - mean) / std


opt_results_aug = []
for r in misses:
    gt_idx = gt_class_index(r["true_label"])
    stem = Path(r["fname"]).stem
    x = preprocess(r["img"]).clone().to(device).requires_grad_(True)

    steps, frames, probs = [], [], []
    for step in range(N_STEPS + 1):
        clean_logits = model(x.unsqueeze(0))
        prob_gt = clean_logits.softmax(dim=1)[0, gt_idx]
        if step % SNAPSHOT_EVERY == 0 or step == N_STEPS:
            frame = tensor_to_pil(x)
            steps.append(step)
            frames.append(frame)
            probs.append(float(prob_gt.detach()))
            save_frame(frame, stem, f"aug_k{N_AUG_SAMPLES}", step)
        if step == N_STEPS:
            break

        # average the gradient over N_AUG_SAMPLES random augmentations (EOT) to get a
        # low-variance estimate of the robust direction, instead of a single noisy sample
        grad = torch.zeros_like(x)
        for _ in range(N_AUG_SAMPLES):
            aug_norm = random_augment(x)
            aug_logits = model(aug_norm.unsqueeze(0))
            grad += torch.autograd.grad(aug_logits[0, gt_idx], x)[0]
        grad /= N_AUG_SAMPLES
        with torch.no_grad():
            x += STEP_SIZE * grad / grad.norm()

    opt_results_aug.append(
        {"fname": r["fname"], "true_label": r["true_label"], "steps": steps, "frames": frames, "probs": probs}
    )
    print(
        f"{r['fname']}: true={r['true_label']!r}  P(true) {probs[0]:.1%} -> {probs[-1]:.1%}  "
        f"(random-augmented EOT-style, {N_STEPS} steps, {N_AUG_SAMPLES} samples/step, shown every {SNAPSHOT_EVERY})"
    )

In [ ]:
n_rows = len(opt_results_aug)
n_steps_cols = len(opt_results_aug[0]["steps"])
n_cols = n_steps_cols + 3  # + initial repeated (start-vs-end), + raw diff, + amplified diff
fig, axes = plt.subplots(n_rows, n_cols, figsize=(1.6 * n_cols, 1.8 * n_rows))
if n_rows == 1:
    axes = axes[None, :]

for row, r in zip(axes, opt_results_aug):
    step_axes = row[:n_steps_cols]
    again_ax, diff_ax, amp_ax = row[n_steps_cols], row[n_steps_cols + 1], row[n_steps_cols + 2]

    for ax, step, frame, prob in zip(step_axes, r["steps"], r["frames"], r["probs"]):
        ax.imshow(frame)
        ax.axis("off")
        ax.set_title(f"step {step}\nP={prob:.0%}", fontsize=8)

    # initial frame shown again right after the final step, so start and end sit
    # side by side instead of at opposite ends of the row
    again_ax.imshow(r["frames"][0])
    again_ax.axis("off")
    again_ax.set_title(f"step {r['steps'][0]} (again)\nP={r['probs'][0]:.0%}", fontsize=8)

    initial = np.asarray(r["frames"][0], dtype=np.float32) / 255.0
    final = np.asarray(r["frames"][-1], dtype=np.float32) / 255.0
    diff = np.abs(final - initial)

    diff_ax.imshow(diff, vmin=0, vmax=1)
    diff_ax.axis("off")
    diff_ax.set_title(f"|Δ| (0-1 scale)\nmax={diff.max():.2f}", fontsize=8)

    d_min, d_max = diff.min(), diff.max()
    amplified = (diff - d_min) / (d_max - d_min + 1e-12)
    amp_ax.imshow(amplified, vmin=0, vmax=1)
    amp_ax.axis("off")
    amp_ax.set_title(f"|Δ| amplified\n[{d_min:.2f}, {d_max:.2f}]→[0,1]", fontsize=8)

    row[0].text(
        -0.15,
        0.5,
        f"{r['true_label']}",
        transform=row[0].transAxes,
        ha="right",
        va="center",
        fontsize=9,
        rotation=0,
    )

fig.suptitle(
    f"EOT-style gradient ascent under random augmentation ({N_STEPS} steps, shown every {SNAPSHOT_EVERY})",
    fontsize=11,
)
fig.tight_layout()
plt.show()
plt.close(fig)